<div dir="rtl" align="right">

# رگرسیون خطی چندگانه

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## ایده در یک نگاه
در دنیای واقعی قیمت خانه فقط به متراژ بستگی ندارد؛ تعداد اتاق، سن بنا و فاصله تا مترو هم مهم‌اند. با چند ویژگی، مدل خطی این شکل را دارد:

$$\hat y=b+w_1x_1+w_2x_2+\dots+w_dx_d$$

با دو ویژگی، مدل به جای خط یک **صفحه** است:

> [با دو ویژگی، مدل خطی یک صفحه در فضای سه‌بعدی است.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/lin_multi.png)

*با دو ویژگی، مدل خطی یک صفحه در فضای سه‌بعدی است.*

## شکل ماتریسی
اگر ستونی از ۱ به داده اضافه کنیم، $X$ ماتریس $n\times(d+1)$ و $\mathbf w$ بردار ضرایب است:

$$\hat{\mathbf y}=X\mathbf w,\qquad \mathbf w^*=(X^\top X)^{-1}X^\top\mathbf y$$

به فرمول دوم **معادله نرمال** می‌گویند. وقتی تعداد ویژگی‌ها خیلی زیاد است، به جای وارون کردن ماتریس از گرادیان کاهشی (درس بعد) استفاده می‌کنیم.

| تعداد ویژگی | روش مناسب |
|---|---|
| کم تا چند هزار | معادله نرمال یا `LinearRegression` |
| خیلی زیاد یا داده خیلی بزرگ | گرادیان کاهشی، `SGDRegressor` |

## تفسیر ضرایب
$w_j$ یعنی: «اگر $x_j$ یک واحد زیاد شود **و بقیه ویژگی‌ها ثابت بمانند**، پیش‌بینی $w_j$ واحد تغییر می‌کند.»

> ضرایب را فقط وقتی با هم مقایسه کن که ویژگی‌ها **مقیاس‌بندی** شده باشند. ضریب «متراژ به متر» و «متراژ به سانتی‌متر» صد برابر فرق دارد ولی اهمیتشان یکی است.

## مثال: داده دیابت
داده Diabetes ده ویژگی دارد (سن، جنس، BMI، فشار خون و شش آزمایش خون) و هدف، پیشرفت بیماری یک سال بعد است.



</div>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

data = load_diabetes(as_frame=True)
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

model = LinearRegression().fit(X_tr, y_tr)
coef = pd.Series(model.coef_, index=X.columns).sort_values()
print(coef.round(1))
print("Test MAE:", round(mean_absolute_error(y_te, model.predict(X_te)), 1))
print("Test R2 :", round(r2_score(y_te, model.predict(X_te)), 3))

# The same answer from the normal equation
A = np.c_[np.ones(len(X_tr)), X_tr]
w = np.linalg.solve(A.T @ A, A.T @ y_tr)
print("Same intercept?", np.isclose(w[0], model.intercept_))

<div dir="rtl" align="right">



## هم‌خطی (Multicollinearity)
اگر دو ویژگی خیلی به هم وابسته باشند (مثلاً متراژ و تعداد اتاق)، مدل نمی‌داند اثر را به کدام بدهد: ضرایب ناپایدار می‌شوند و حتی علامتشان برعکس می‌شود، هرچند پیش‌بینی خوب می‌ماند.

| نشانه | ابزار تشخیص | راه حل |
|---|---|---|
| ضرایب با تغییر کوچک داده عوض می‌شوند | ماتریس همبستگی | حذف یکی از ویژگی‌ها |
| علامت ضریب غیرمنطقی است | VIF بزرگ‌تر از ۵ تا ۱۰ | Ridge (درس‌های بعد) |
| | | PCA |



</div>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

X = load_diabetes(as_frame=True).data
print(X.corr().round(2).loc[["s1", "s2", "s3", "s4"], ["s1", "s2", "s3", "s4"]])

# Variance inflation factor: VIF_j = 1 / (1 - R2 of x_j predicted from the other features)
vif = {}
for col in X.columns:
    others = X.drop(columns=col)
    r2 = LinearRegression().fit(others, X[col]).score(others, X[col])
    vif[col] = 1 / (1 - r2)
print(pd.Series(vif).round(1).sort_values(ascending=False))

<div dir="rtl" align="right">



## ویژگی‌های دسته‌ای
ستونی مثل «شهر» عدد نیست. با **One-Hot** برای هر شهر یک ستون ۰/۱ می‌سازیم. یکی از ستون‌ها را حذف می‌کنیم (`drop="first"`) تا هم‌خطی کامل پیش نیاید.



</div>

In [ ]:
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({
    "size": [80, 120, 95, 150, 60, 110],
    "city": ["Tehran", "Shiraz", "Tehran", "Tabriz", "Shiraz", "Tabriz"],
    "price": [9.5, 7.8, 10.6, 9.0, 4.1, 6.9],
})
prep = make_column_transformer((OneHotEncoder(drop="first"), ["city"]), remainder="passthrough")
model = make_pipeline(prep, LinearRegression()).fit(df[["size", "city"]], df["price"])
print(prep.get_feature_names_out())
print(model[-1].coef_.round(3))
print(model.predict(pd.DataFrame({"size": [100], "city": ["Tehran"]})).round(2))

<div dir="rtl" align="right">

### تمرین

در داده دیابت فقط با سه ویژگی bmi، bp و s5 مدل بساز. R² آزمون چقدر کم می‌شود؟

</div>

In [ ]:
# Your answer here